In [14]:
from langgraph.graph import START , END , StateGraph
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
from dotenv import load_dotenv
load_dotenv()

## InMemorySaver is used to save the state in memory RAM. Its not used in production application its used for demo purpose

True

In [15]:
class Joke(TypedDict):
    topic : str
    joke : str
    explanation : str

model = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')

In [16]:
def generate_joke(state: Joke):
    prompt = f"Generate a joke on the given topic: {state['topic']}"
    state['joke'] = model.invoke(prompt).content
    return {'joke':state['joke']}

In [17]:
def generate_explanation(state: Joke):
    prompt = f"Generate a explanation of the given joke: {state['joke']}"
    state['explanation'] = model.invoke(prompt).content
    return {'explanation':state['explanation']}

In [18]:
graph = StateGraph(Joke)
graph.add_node('generate_joke' , generate_joke)
graph.add_node('generate_explanation' , generate_explanation)

graph.add_edge(START , 'generate_joke')
graph.add_edge('generate_joke' , 'generate_explanation')
graph.add_edge('generate_explanation' , END)

In [ ]:
checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)
config1 = {'configurable':{'thread_id':'1'}}

## thread is used to assign a number to a workflow to access it further
## We can assign different thread to diff input
print(workflow.invoke({'topic':'samosa'},config=config1))

{'topic': 'samosa', 'joke': "Here's a joke about samosas:\n\nWhy did the samosa break up with the spring roll?\n\nBecause he felt like he was always getting *wrapped up* in their relationship, and she just wasn't *filled* with enough commitment!", 'explanation': 'This joke plays on a few puns related to the ingredients and preparation of samosas and spring rolls. Here\'s a breakdown:\n\n*   **"Wrapped up"**:\n    *   **Literal meaning for spring rolls:** Spring rolls are literally *wrapped* in a thin pastry or wrapper.\n    *   **Figurative meaning for relationships:** "Wrapped up" in a relationship means to be very involved, perhaps too involved, or feeling trapped by it. The samosa is saying he felt too consumed or overwhelmed by the relationship.\n\n*   **"Filled with enough commitment"**:\n    *   **Literal meaning for samosas:** Samosas are *filled* with a savory mixture (like potatoes, peas, and spices).\n    *   **Figurative meaning for relationships:** "Filled with commitment" 

In [ ]:
## To get the final state value of thread id 1 again
workflow.get_state(config1)

StateSnapshot(values={'topic': 'samosa', 'joke': "Here's a joke about samosas:\n\nWhy did the samosa break up with the spring roll?\n\nBecause he felt like he was always getting *wrapped up* in their relationship, and she just wasn't *filled* with enough commitment!", 'explanation': 'This joke plays on a few puns related to the ingredients and preparation of samosas and spring rolls. Here\'s a breakdown:\n\n*   **"Wrapped up"**:\n    *   **Literal meaning for spring rolls:** Spring rolls are literally *wrapped* in a thin pastry or wrapper.\n    *   **Figurative meaning for relationships:** "Wrapped up" in a relationship means to be very involved, perhaps too involved, or feeling trapped by it. The samosa is saying he felt too consumed or overwhelmed by the relationship.\n\n*   **"Filled with enough commitment"**:\n    *   **Literal meaning for samosas:** Samosas are *filled* with a savory mixture (like potatoes, peas, and spices).\n    *   **Figurative meaning for relationships:** "Fil

In [21]:
## To get the complete history of state on every node
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': "Here's a joke about samosas:\n\nWhy did the samosa break up with the spring roll?\n\nBecause he felt like he was always getting *wrapped up* in their relationship, and she just wasn't *filled* with enough commitment!", 'explanation': 'This joke plays on a few puns related to the ingredients and preparation of samosas and spring rolls. Here\'s a breakdown:\n\n*   **"Wrapped up"**:\n    *   **Literal meaning for spring rolls:** Spring rolls are literally *wrapped* in a thin pastry or wrapper.\n    *   **Figurative meaning for relationships:** "Wrapped up" in a relationship means to be very involved, perhaps too involved, or feeling trapped by it. The samosa is saying he felt too consumed or overwhelmed by the relationship.\n\n*   **"Filled with enough commitment"**:\n    *   **Literal meaning for samosas:** Samosas are *filled* with a savory mixture (like potatoes, peas, and spices).\n    *   **Figurative meaning for relationships:** "Fi

In [ ]:
## if a node get crashed , so to resume it
workflow.invoke(None , config=config1)

In [ ]:
## To perform time travel (To start again the workflow from any node or checkpoint)
workflow.invoke(None , {"configurable":{"thread_id":"1" , "checkpoint_id":""}}) ## Add that checkpoint id

In [23]:
## To update any state
workflow.invoke({"topic": "Pizza"},{"configurable": {"thread_id": "1","checkpoint_id": "1f150705-648e-633c-8000-1cbd1509768d"}})

{'topic': 'Pizza',
 'joke': 'Why did the pizza get a job at the bakery?\n\nBecause it kneaded the dough!',
 'explanation': 'Here\'s an explanation of that joke:\n\nThe joke is a pun, which means it plays on words that sound alike but have different meanings.\n\n*   **"Kneaded the dough"** has a double meaning:\n    *   **Literal meaning:** When you make pizza dough, you have to "knead" it. This is a physical process of pushing and folding the dough to develop gluten.\n    *   **Figurative meaning:** "To need the dough" sounds exactly the same as "to knead the dough." In this context, "dough" is slang for **money**.\n\n**Therefore, the punchline works because:**\n\n*   Pizza dough needs to be **kneaded** (the action).\n*   The pizza **needed the dough** (money) to get a job.\n\nThe humor comes from the unexpected twist of applying the literal action of making pizza to the figurative meaning of needing money for employment.'}